# Gemini Omni Flash: Generate and Edit Videos
**by Navin Reddy | Telusko**

---

In this notebook you will learn how to:
- Generate videos from a text prompt
- Control aspect ratio and duration
- Animate a still image into a video
- Combine multiple reference images into one video scene
- Edit any generated video using plain English, in a multi-turn conversation

All of this uses the **Interactions API** from Google. The key idea: every video gets an ID. Pass that ID into the next call to edit it. That is what makes Omni different from every other video AI.


## Step 1: Install the SDK

In [ ]:
%pip install -U -q "google-genai>=2.8.0"

## Step 2: Add Your API Key

1. Go to [aistudio.google.com](https://aistudio.google.com) and click **Get API Key**
2. In Colab, click the key icon in the left sidebar (Secrets)
3. Add a secret named `GEMINI_API_KEY` and paste your key
4. Toggle **Notebook access** ON


In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

print(" Client ready")

 Client ready


## Step 3: Configuration and Helper Functions

Two models are used in this notebook:
- `IMAGE_MODEL` generates still images (used as input for video)
- `OMNI_MODEL` is the Omni Flash video model


In [ ]:
from IPython.display import display, HTML

IMAGE_MODEL  = "gemini-2.5-flash-image"
OMNI_MODEL   = "gemini-omni-flash-preview
ASPECT_RATIO = "16:9"

IMAGE_CONFIG = {"generation_config": {"image_config": {"aspect_ratio": ASPECT_RATIO}}}

generated_videos = []
last_interaction_id = None

def get_output_image(interaction):
    """Retrieve the last image generated from the interaction steps."""
    if not hasattr(interaction, 'steps') or not interaction.steps:
        return None
    for step in reversed(interaction.steps):
        if step.type == "model_output" and step.content:
            for content in reversed(step.content):
                if content.type == "image":
                    return content
    return None

def get_output_video(interaction):
    """Retrieve the last video generated from the interaction steps."""
    if not hasattr(interaction, 'steps') or not interaction.steps:
        return None
    for step in reversed(interaction.steps):
        if step.type == "model_output" and step.content:
            for content in reversed(step.content):
                if content.type == "video":
                    return content
    return None

def show_image(output):
    """Render an image output inline."""
    if output is None:
        return
    if hasattr(output, 'steps'):
        output = get_output_image(output)
    if output is None:
        print(" No image content found in interaction steps")
        return
    mime = getattr(output, 'mime_type', 'image/png') or 'image/png'
    display(HTML(f'<img src="data:{mime};base64,{output.data}" style="max-width:512px; border-radius:8px;" />'))

def show_video(output):
    """Render a video output inline."""
    if output is None:
        return
    if hasattr(output, 'steps'):
        output = get_output_video(output)
    if output is None:
        print(" No video content found in interaction steps")
        return
    mime = getattr(output, 'mime_type', 'video/mp4') or 'video/mp4'
    display(HTML(
        f'<video controls style="max-width:640px; border-radius:8px;">'
        f'<source src="data:{mime};base64,{output.data}" type="{mime}">'
        f'</video>'
    ))

print(f" Setup complete. Image model: {IMAGE_MODEL}, Omni model: {OMNI_MODEL}")

 Setup complete. Image model: gemini-2.5-flash-image, Omni model: bouncybohr


## Feature 1: Text to Video

Give the model a text prompt and set `response_format` to `video`.


In [ ]:
prompt = "A lone astronaut floats in deep space, slowly rotating, Earth glowing blue far below, Milky Way stretching across the entire background, suit reflecting starlight, complete silence, cinematic IMAX wide shot, photorealistic"  # @param {type:"string"}

interaction = client.interactions.create(
    model=OMNI_MODEL,
    input=prompt,
    background=False,
    store=False,
    stream=False,
    extra_body={"response_format": {"type": "video"}},
)

out_video = get_output_video(interaction)
if out_video:
    show_video(out_video)
    generated_videos.append(out_video)
else:
    print(" No video generated")

## Feature 2: Portrait Video (9:16)

Add `aspect_ratio` to the response format to control orientation.
Use `9:16` for YouTube Shorts and Instagram Reels.


In [ ]:
res_prompt = "An astronaut helmet in extreme close-up, visor reflecting a full nebula in purple and gold, stars slowly drifting, breath fogging the inner glass, vertical cinematic frame, photorealistic, IMAX quality"  # @param {type:"string"}
ASPECT_RATIO_9_16 = "9:16"  # @param ["9:16", "16:9"]

print(f"Generating video with Aspect Ratio: {ASPECT_RATIO_9_16}")

video_config = {
    "type": "video",
    "aspect_ratio": ASPECT_RATIO_9_16,
}

interaction_res = client.interactions.create(
    model=OMNI_MODEL,
    input=res_prompt,
    response_format=video_config
)

out_vid = get_output_video(interaction_res)
if out_vid:
    show_video(out_vid)
    generated_videos.append(out_vid)
else:
    print(" No video generated")

## Feature 3: Duration Control

Omni supports `5s` or `10s` video length.


In [ ]:
duration_prompt = "A space shuttle launches from Earth, fire and smoke exploding beneath, rising through clouds, breaking into clear blue sky, atmosphere thinning to black, stars appearing one by one, single continuous upward shot"  # @param {type:"string"}
DURATION_SECONDS = "5s"  # @param ["5s", "10s"]

print(f"Generating video with duration: {DURATION_SECONDS}")

video_config_dur = {
    "type": "video",
    "duration": DURATION_SECONDS,
}

interaction = client.interactions.create(
    model=OMNI_MODEL,
    input=duration_prompt,
    background=False,
    store=False,
    stream=False,
    response_format=video_config_dur
)

out_vid = get_output_video(interaction)
if out_vid:
    show_video(out_vid)
    generated_videos.append(out_vid)
else:
    print(" No video generated")

## Feature 4: Image to Video

Two steps:
1. Generate a still image using the image model
2. Pass that image to Omni with a motion description to animate it


In [ ]:
  image_prompt = "An abandoned space station interior, emergency red lighting, floating debris drifting slowly, a single porthole window showing a massive ringed planet outside, photorealistic, cinematic, ultra detailed"  # @param {type:"string"}

print("Generating source image...")

image_format = {
    "type": "image",
    "aspect_ratio": ASPECT_RATIO,
}

img_interaction = client.interactions.create(
    model=IMAGE_MODEL,
    input=image_prompt,
    background=False,
    store=False,
    stream=False,
    extra_body={"response_format": image_format},
)
ref_image = get_output_image(img_interaction)

if not ref_image:
    print(" No image generated")
else:
    show_image(ref_image)

In [ ]:
video_prompt = "Debris drifts silently past the porthole, the ringed planet slowly rotates in the distance, red emergency lights flicker once, a single oxygen tank tumbles past the camera in slow motion"  # @param {type:"string"}

print("Animating image...")

video_format = {
    "type": "video",
    "aspect_ratio": ASPECT_RATIO,
}

interaction = client.interactions.create(
    model=OMNI_MODEL,
    input=[
        {"type": "image", "data": ref_image.data, "mime_type": ref_image.mime_type or "image/png"},
        {"type": "text", "text": video_prompt}
    ],
    extra_body={"response_format": video_format},
    background=False,
    store=False,
    stream=False,
)

out_vid = get_output_video(interaction)
if out_vid:
    show_video(out_vid)
    generated_videos.append(out_vid)
else:
    print(" No video generated")

## Feature 5: Reference to Video

Generate two separate images, then ask Omni to create a scene combining both.

The first image uses `store=True` and is referenced by its interaction ID.
The second image is passed inline as base64.


In [ ]:
ref_prompt_1 = "A detailed astronaut in a worn NASA spacesuit, helmet on, floating in zero gravity, photorealistic, dark space background, studio lit"  # @param {type:"string"}
ref_prompt_2 = "A glowing golden wormhole portal open in deep space, swirling energy rings, stars bending around the edges, photorealistic"  # @param {type:"string"}

print("Generating reference images...")

image_format = {
    "type": "image",
    "aspect_ratio": ASPECT_RATIO,
}

r1 = client.interactions.create(
    model=IMAGE_MODEL,
    input=ref_prompt_1,
    background=False,
    store=True,
    stream=False,
    extra_body={"response_format": image_format},
)
print(f"Reference 1 Interaction ID (stored): {r1.id}")
ref_agent = get_output_image(r1)
if not ref_agent:
    print(" No image generated")
else:
    show_image(ref_agent)

r2 = client.interactions.create(
    model=IMAGE_MODEL,
    input=ref_prompt_2,
    background=False,
    store=False,
    stream=False,
    extra_body={"response_format": image_format},
)
ref_desk = get_output_image(r2)
if not ref_desk:
    print(" No image generated")
else:
    show_image(ref_desk)

In [ ]:
video_prompt = "The astronaut drifts slowly toward the glowing wormhole, reaches out one hand as golden light washes across the suit, camera pushes in dramatically, cinematic silence, single continuous shot"  # @param {type:"string"}

print("Generating video with references...")

video_format = {
    "type": "video",
    "aspect_ratio": ASPECT_RATIO,
}

interaction = client.interactions.create(
    model=OMNI_MODEL,
    previous_interaction_id=r1.id,
    input=[
        {"type": "image", "data": ref_desk.data, "mime_type": ref_desk.mime_type or "image/png"},
        {"type": "text", "text": video_prompt}
    ],
    background=False,
    store=True,
    stream=False,
    extra_body={"response_format": video_format},
)

print(f"Interaction ID: {interaction.id}")
last_interaction_id = interaction.id

out_video = get_output_video(interaction)
if out_video:
    show_video(out_video)
    generated_videos.append(out_video)
else:
    print(" No video generated")

## Feature 6: Multi-turn Video Editing

Pass `previous_interaction_id` with a plain English edit instruction.
Omni looks up the stored video and applies your change — no re-uploading needed.

Every edit produces a new interaction ID. The chain continues from `last_interaction_id`.


### Video Editing (character)

Apply a targeted change to an element in the video.


In [ ]:
edit_prompt = "Change the astronaut suit color from white NASA to a sleek black and gold private space company design, keep every motion and scene element exactly the same"  # @param {type:"string"}

if last_interaction_id:
    print("Editing video...")

    video_format = {
        "type": "video",
        "aspect_ratio": ASPECT_RATIO,
    }

    interaction = client.interactions.create(
        model=OMNI_MODEL,
        previous_interaction_id=last_interaction_id,
        input=edit_prompt,
        background=False,
        store=True,
        stream=False,
        extra_body={"response_format": video_format},
    )
    print(f"Interaction ID: {interaction.id}")
    last_interaction_id = interaction.id

    out_video = get_output_video(interaction)
    if out_video:
        show_video(out_video)
        generated_videos.append(out_video)
    else:
        print(" No video generated")
else:
    print(" No last interaction ID found. Run Feature 5 first.")

### Video Editing (style)

Apply a full visual style change to the entire scene.


In [ ]:
style_prompt = "Transform the entire scene into a painted impressionist style, thick brushstrokes, Van Gogh Starry Night color palette of deep blue and swirling gold, keep all motion and timing exactly the same"  # @param {type:"string"}

if last_interaction_id:
    print("Styling video...")

    video_format = {
        "type": "video",
        "aspect_ratio": ASPECT_RATIO,
    }

    interaction = client.interactions.create(
        model=OMNI_MODEL,
        previous_interaction_id=last_interaction_id,
        input=style_prompt,
        background=False,
        store=True,
        stream=False,
        extra_body={"response_format": video_format},
    )
    print(f"Interaction ID: {interaction.id}")
    last_interaction_id = interaction.id

    out_video = get_output_video(interaction)
    if out_video:
        show_video(out_video)
        generated_videos.append(out_video)
    else:
        print(" No video generated")
else:
    print(" No last interaction ID found. Run Feature 5 first.")

## Feature 7: Video Editing with a Reference Image

Generate a reference image and pass it alongside your edit instruction.
The same reference image can be used two ways:
- Restyle the whole video to match the reference
- Insert the reference as an object inside the scene


In [ ]:
style_prompt = "A vintage 1960s NASA mission patch design, circular badge, rocket and stars, bold flat illustration, deep navy and gold, retro space program aesthetic"  # @param {type:"string"}

print("Generating style reference...")

image_format = {
    "type": "image",
    "aspect_ratio": ASPECT_RATIO,
}

r = client.interactions.create(
    model=IMAGE_MODEL,
    input=style_prompt,
    background=False,
    store=False,
    stream=False,
    extra_body={"response_format": image_format},
)

ref_style = get_output_image(r)
if not ref_style:
    print(" No style reference image generated")
else:
    show_image(ref_style)

In [ ]:
restyle_prompt = "Restyle the entire video to match this vintage NASA mission patch illustration style, flat bold colors, retro space aesthetic, keep all motion and composition exactly the same"  # @param {type:"string"}

if last_interaction_id and 'ref_style' in locals() and ref_style:
    print("Restyling video with reference...")

    video_format = {
        "type": "video",
        "aspect_ratio": ASPECT_RATIO,
    }

    interaction = client.interactions.create(
        model=OMNI_MODEL,
        previous_interaction_id=last_interaction_id,
        input=[
            {"type": "image", "data": ref_style.data, "mime_type": ref_style.mime_type or "image/png"},
            {"type": "text", "text": restyle_prompt}
        ],
        background=False,
        store=True,
        stream=False,
        extra_body={"response_format": video_format},
    )

    out_video = get_output_video(interaction)
    if out_video:
        show_video(out_video)
        generated_videos.append(out_video)
    else:
        print(" No video generated")
else:
    print(" Missing previous interaction ID or reference style image.")

In [ ]:
restyle_prompt = "Place this vintage mission patch as an emblem on the astronaut suit shoulder, glowing softly, stitched texture, perfectly integrated with the suit material and lighting"  # @param {type:"string"}

if last_interaction_id and 'ref_style' in locals() and ref_style:
    print("Inserting reference into scene...")

    video_format = {
        "type": "video",
        "aspect_ratio": ASPECT_RATIO,
    }

    interaction = client.interactions.create(
        model=OMNI_MODEL,
        previous_interaction_id=last_interaction_id,
        input=[
            {"type": "image", "data": ref_style.data, "mime_type": ref_style.mime_type or "image/png"},
            {"type": "text", "text": restyle_prompt}
        ],
        background=False,
        store=True,
        stream=False,
        extra_body={"response_format": video_format},
    )
    print(f"Interaction ID: {interaction.id}")
    last_interaction_id = interaction.id

    out_video = get_output_video(interaction)
    if out_video:
        show_video(out_video)
        generated_videos.append(out_video)
    else:
        print(" No video generated")
else:
    print(" Missing previous interaction ID or reference style image.")

## Feature 8: Edit Your Own Video

Upload any mp4 file and ask Omni to edit it.

Steps:
1. Upload your video file using the Colab file browser (left sidebar, folder icon)
2. Set the filename below to match your uploaded file
3. Run this cell

The cell uploads the file, waits for Google to process it, sends the edit instruction, shows the result, and deletes the uploaded file to free up quota.


In [ ]:
import time

VIDEO_FILENAME = "Video.mp4"  # @param {type:"string"}
edit_instruction = "Add a soft glowing aurora effect in the deep space background behind the astronauts, subtle green and purple light waves drifting slowly, keep all existing motion and characters exactly the same"  # @param {type:"string"}

print(f"Uploading {VIDEO_FILENAME}...")
video_file = client.files.upload(file=VIDEO_FILENAME)

while video_file.state == "PROCESSING":
    print("Waiting for video to be processed...")
    time.sleep(10)
    video_file = client.files.get(name=video_file.name)

if video_file.state == "FAILED":
    raise ValueError(f"Video processing failed: {video_file.state}")

print(f"Video ready: {video_file.uri}")

video_format = {
    "type": "video",
    "aspect_ratio": ASPECT_RATIO,
}

interaction = client.interactions.create(
    model=OMNI_MODEL,
    input=[
        {"type": "document", "uri": video_file.uri},
        {"type": "text",     "text": edit_instruction}
    ],
    extra_body={"response_format": video_format},
    background=False,
    store=False,
    stream=False,
)

out_vid = get_output_video(interaction)
if out_vid:
    show_video(out_vid)
    generated_videos.append(out_vid)
else:
    print("No video generated")

client.files.delete(name=video_file.name)
print("Uploaded file deleted")

## Save All Videos to Google Drive


In [ ]:
from google.colab import drive
import base64, os

drive.mount("/content/drive")

save_folder = "/content/drive/MyDrive/Telusko_Omni_Videos"
os.makedirs(save_folder, exist_ok=True)

for i, video in enumerate(generated_videos):
    video_bytes = base64.b64decode(video.data)
    path = f"{save_folder}/omni_video_{i+1}.mp4"
    with open(path, "wb") as f:
        f.write(video_bytes)
    print(f"Saved: {path}")

print(f"\n All {len(generated_videos)} videos saved to Google Drive")

## What You Learned

In this notebook you used the **Gemini Omni Flash Interactions API** to:

| Feature | What it does |
|---|---|
| Text to Video | Generate a video from a text prompt |
| Portrait mode | Control aspect ratio for Shorts and Reels |
| Duration control | Choose between 5s and 10s video length |
| Image to Video | Animate a generated still image |
| Reference to Video | Combine two images into one video scene |
| Multi-turn editing | Edit any video with plain English, chained by interaction ID |
| Reference-based editing | Use an image to guide a style change or object insertion |

**The key concept:** every call returns an interaction ID. Pass that ID into the next call as `previous_interaction_id`. That is the chain that makes iterative video editing possible.

---

Subscribe to Telusko for more AI and Java content: [youtube.com/@telusko](https://youtube.com/@telusko)

Hello Aliens. Keep building.
